# 1회차 실습 — 벡터·노름·내적·코사인 유사도 (MNIST)

> Part 1 — 1회차 (벡터·선형결합·노름·내적·코사인 유사도)
> 사전 reading: MML §2.1, §3.1, §3.2 / Strang §1.1, §1.2 / 3Blue1Brown EoLA Ch.1-2

## 학습 목표

1. NumPy로 벡터의 덧셈·스칼라곱·선형결합을 구현한다.
2. $\ell_1, \ell_2, \ell_\infty$ 노름을 **직접 구현**하고 `np.linalg.norm`과 결과를 비교한다.
3. 내적과 코사인 유사도를 직접 구현한다.
4. MNIST 두 이미지를 $\mathbb{R}^{784}$ 벡터로 변환 후 코사인 유사도를 계산한다.
5. 한 query 이미지의 top-5 nearest neighbor를 코사인 유사도로 검색한다.

## 사용 라이브러리

- NumPy, Matplotlib (필수)
- scikit-learn (MNIST 일부 로드용)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print('NumPy version:', np.__version__)

## 1. 벡터의 기본 연산

$\mathbb{R}^4$의 두 벡터:

$$\mathbf{a} = (3, 1, -2, 4)^\top, \quad \mathbf{b} = (-1, 2, 5, 0)^\top$$

In [ ]:
a = np.array([3, 1, -2, 4], dtype=float)
b = np.array([-1, 2, 5, 0], dtype=float)

print('a =', a)
print('b =', b)
print('a + b =', a + b)
print('2a - 3b =', 2*a - 3*b)  # 선형결합 예제

### 1.1 선형결합 — 행렬·벡터 곱의 미리보기

두 열벡터 $\mathbf{v}_1, \mathbf{v}_2$를 한 행렬의 열로 묶고 스칼라 $(x_1, x_2)$를 곱하면 **두 열의 선형결합**이 나온다. 2회차에서 본격 다룬다.

In [ ]:
v1 = np.array([1, 0, 0])
v2 = np.array([0, 1, 0])
x = np.array([2, -3])

# 직접 선형결합
lc_direct = x[0] * v1 + x[1] * v2
print('직접 선형결합:', lc_direct)

# 행렬·벡터 곱으로 표현
A = np.column_stack([v1, v2])  # 열로 묶은 행렬
lc_matrix = A @ x
print('A @ x:', lc_matrix)

assert np.allclose(lc_direct, lc_matrix), '두 방식 결과가 일치해야 함'
print('✓ 선형결합 = 행렬·벡터 곱 (열 방식 해석)')

## 2. 노름 — 직접 구현

정의:
- $\|\mathbf{x}\|_1 = \sum_i |x_i|$
- $\|\mathbf{x}\|_2 = \sqrt{\sum_i x_i^2}$
- $\|\mathbf{x}\|_\infty = \max_i |x_i|$

**`np.linalg.norm`을 호출하지 않고** 직접 구현한 뒤 비교한다.

In [ ]:
def my_norm(v, p=2):
    """p-노름 직접 구현. p=1, 2, np.inf 지원.
    NumPy 기본 연산만 사용 (np.linalg.norm 금지).
    """
    v = np.asarray(v, dtype=float)
    if p == 1:
        return np.abs(v).sum()
    if p == 2:
        return float(np.sqrt((v ** 2).sum()))
    if p == np.inf:
        return float(np.abs(v).max())
    # 일반 p
    return float((np.abs(v) ** p).sum() ** (1.0 / p))


# 검증
for p in [1, 2, np.inf]:
    mine = my_norm(a, p)
    lib = np.linalg.norm(a, p)
    print(f'p={p}:  mine={mine:.6f}, lib={lib:.6f}, match={np.isclose(mine, lib)}')

### 2.1 단위구 시각화 ($\mathbb{R}^2$)

$\ell_1, \ell_2, \ell_\infty$ 단위구를 그려서 형태를 확인한다.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
circle = np.stack([np.cos(theta), np.sin(theta)])  # 2 × 400, 단위 원

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, p, title in zip(axes, [1, 2, np.inf], ['L1 (마름모)', 'L2 (원)', 'L_inf (정사각형)']):
    # 단위 원의 각 점을 그 점의 노름으로 정규화 → 해당 p-노름 단위구의 점
    norms = np.array([my_norm(circle[:, i], p) for i in range(circle.shape[1])])
    pts = circle / norms
    ax.plot(pts[0], pts[1])
    ax.set_aspect('equal')
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
plt.suptitle('p-노름의 단위구 ($\mathbb{R}^2$)')
plt.tight_layout()
plt.show()

print('관찰: L1의 꼭짓점이 좌표축 위에 있어서 sparse 해를 유도. L2는 모든 방향 균등.')

## 3. 내적과 코사인 유사도

정의:
$$\mathbf{u}^\top \mathbf{v} = \sum_i u_i v_i, \qquad \cos\theta = \frac{\mathbf{u}^\top \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

In [ ]:
def my_dot(u, v):
    """내적 직접 구현 — for문 없는 numpy 연산만"""
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return float((u * v).sum())


def my_cosine_similarity(u, v):
    """코사인 유사도 직접 구현"""
    return my_dot(u, v) / (my_norm(u, 2) * my_norm(v, 2))


# a, b의 내적과 코사인 유사도
print(f'a · b = {my_dot(a, b):.4f}')
print(f'cos(a, b) = {my_cosine_similarity(a, b):.6f}')
print(f'각도 = {np.degrees(np.arccos(my_cosine_similarity(a, b))):.2f}도 → 둔각')

# np.dot과 비교
assert np.isclose(my_dot(a, b), np.dot(a, b))
print('✓ my_dot == np.dot')

### 3.1 코사인 유사도의 스케일 불변성 확인

$\cos(\alpha\mathbf{u}, \beta\mathbf{v}) = \cos(\mathbf{u}, \mathbf{v})$ for $\alpha, \beta > 0$

In [ ]:
u = np.array([1, 1, 0, 0], dtype=float)
v = np.array([10, 10, 0, 0], dtype=float)

print(f'cos(u, v) = {my_cosine_similarity(u, v):.6f}  (방향 같음 → 1)')
print(f'유클리드 거리 |u - v|_2 = {my_norm(u - v, 2):.4f}  (길이 차이로 큰 거리)')
print()
print('교훈: 문서 길이가 다르면 유클리드는 멀게 보지만 코사인은 같다고 본다.')

## 4. MNIST — 이미지를 벡터로

MNIST 손글씨 숫자 데이터는 28×28 흑백 이미지. **flatten해서 $\mathbb{R}^{784}$의 벡터로 본다.**

여기서는 `sklearn.datasets.load_digits()`의 8×8 mini-MNIST를 쓴다 (다운로드 없음, 항상 사용 가능).
원본 28×28 MNIST를 쓰려면 `tensorflow.keras.datasets.mnist.load_data()` 또는 `torchvision.datasets.MNIST` 사용.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_img = digits.images   # (1797, 8, 8)
X = digits.data         # (1797, 64) — 이미 flatten된 형태
y = digits.target       # (1797,)

print(f'이미지 개수: {X_img.shape[0]}')
print(f'이미지 크기: {X_img.shape[1]} x {X_img.shape[2]}')
print(f'flatten 벡터 차원: {X.shape[1]}')
print(f'레이블 종류: {np.unique(y)}')

In [ ]:
# 두 이미지 시각화 + 코사인 유사도
idx1, idx2 = 0, 100  # 임의의 두 이미지

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(X_img[idx1], cmap='gray_r')
axes[0].set_title(f'index={idx1}, label={y[idx1]}')
axes[0].axis('off')
axes[1].imshow(X_img[idx2], cmap='gray_r')
axes[1].set_title(f'index={idx2}, label={y[idx2]}')
axes[1].axis('off')
plt.show()

sim = my_cosine_similarity(X[idx1], X[idx2])
print(f'\ncos similarity between idx={idx1} and idx={idx2}: {sim:.4f}')
print(f'두 라벨 같음? {y[idx1] == y[idx2]}')

### 4.1 같은 숫자 vs 다른 숫자의 평균 코사인 유사도

같은 라벨 쌍과 다른 라벨 쌍의 평균을 비교.

In [ ]:
# 작은 표본으로 평균 비교 (n=200으로 제한 → 200×200 행렬 안정)
n_sample = 200
X_s = X[:n_sample]
y_s = y[:n_sample]

# 코사인 유사도 행렬 (broadcasting 버전)
norms = np.sqrt((X_s ** 2).sum(axis=1, keepdims=True))  # (n, 1)
X_norm = X_s / (norms + 1e-12)                          # 행별 정규화
cos_mat = X_norm @ X_norm.T                             # (n, n)

# 같은 라벨 쌍 vs 다른 라벨 쌍
same_label = y_s[:, None] == y_s[None, :]
off_diag = ~np.eye(n_sample, dtype=bool)

same_mean = cos_mat[same_label & off_diag].mean()
diff_mean = cos_mat[~same_label].mean()

print(f'같은 라벨 쌍 평균 cos: {same_mean:.4f}')
print(f'다른 라벨 쌍 평균 cos: {diff_mean:.4f}')
print(f'차이: {same_mean - diff_mean:.4f}  → 같은 숫자일수록 더 가깝다 (검증)')

## 5. Top-5 Nearest Neighbor 검색

한 query 이미지가 주어졌을 때, **코사인 유사도가 가장 높은 상위 5개** 이미지를 찾고 시각화.

In [ ]:
# query 선정
query_idx = 7
query_vec = X[query_idx]
query_label = y[query_idx]

# 모든 이미지와 query의 코사인 유사도 계산 (broadcasting)
norms_all = np.sqrt((X ** 2).sum(axis=1))
query_norm = np.sqrt((query_vec ** 2).sum())
sims = (X @ query_vec) / (norms_all * query_norm + 1e-12)

# 자기 자신 제외하고 top-5
sims_copy = sims.copy()
sims_copy[query_idx] = -np.inf
top5_idx = np.argsort(-sims_copy)[:5]

# 시각화
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
axes[0].imshow(X_img[query_idx], cmap='gray_r')
axes[0].set_title(f'Query (label={query_label})')
axes[0].axis('off')
for ax, idx in zip(axes[1:], top5_idx):
    ax.imshow(X_img[idx], cmap='gray_r')
    ax.set_title(f'label={y[idx]}\ncos={sims[idx]:.3f}')
    ax.axis('off')
plt.tight_layout()
plt.show()

hit = (y[top5_idx] == query_label).sum()
print(f'\nQuery label {query_label} 기준 top-5 중 같은 라벨 {hit}/5')

## 6. 연습 (자가 점검)

다음을 직접 시도해 보고 결과를 본인 노트에 기록한다.

### 연습 1 (Top-5의 정확도)
`query_idx`를 0, 1, ..., 9로 바꿔가며 top-5 중 같은 라벨의 비율을 측정. 표로 정리.

### 연습 2 (정규화 효과)
코사인 유사도 대신 **유클리드 거리** 기준으로 top-5를 찾아보고 결과 차이를 비교.

### 연습 3 (broadcasting vs for문 시간 비교)
$n = 10, 100, 1000$인 무작위 행렬 $X \in \mathbb{R}^{n \times 30}$에 대해
1. 이중 for문으로 $n \times n$ 코사인 유사도 행렬 계산
2. broadcasting으로 같은 결과 계산 (위 셀의 `X_norm @ X_norm.T` 방식)

두 방식의 시간을 측정해 log-log 그래프로 그리고 broadcasting의 speedup을 확인한다.

### 연습 4 (코시-슈바르츠 부등식 수치 검증)
1000회 임의의 $\mathbf{u}, \mathbf{v} \in \mathbb{R}^5$를 생성하여
$|\mathbf{u}^\top \mathbf{v}| \le \|\mathbf{u}\| \|\mathbf{v}\|$가 항상 성립함을 확인하고, 등호가 성립하는 경우를 만들어보라 ($\mathbf{v} = c\mathbf{u}$).

→ 정답·풀이는 과제 1회차 제출본에 포함한다.

## 7. 정리

오늘 검증한 사실:

- 노름·내적·코사인 유사도는 **NumPy 기본 연산만으로 한 줄씩 구현 가능**
- 단위구 모양은 $p$에 따라 마름모·원·정사각형으로 달라짐
- 코사인 유사도는 **스케일 불변** — 길이 다른 문서·이미지 비교에 자연스러움
- MNIST에서 **같은 라벨끼리 코사인 유사도가 평균적으로 더 높다** (간단한 nearest neighbor 분류기의 토대)
- broadcasting + 행렬곱(`X_norm @ X_norm.T`) 한 줄로 모든 쌍의 코사인 유사도 행렬을 얻을 수 있다 — 12회차 attention의 $QK^\top$와 같은 패턴

### 다음 회차로 이어지는 관찰

이번 실습에서 우리는 이미 다음 두 가지를 했다:
1. **행렬·벡터 곱을 열의 선형결합으로 본다** (셀 1.1) — 2회차의 주제
2. **행렬·행렬 곱 $X_{norm} X_{norm}^\top$이 모든 쌍의 내적을 한 번에 계산한다** — 12회차 attention의 핵심

1회차가 25회차 전체의 첫 단추가 되는 셈이다.